# Example: Comparing Frequency Response Estimation Methods

This example compares the three frequency-domain estimators in sid:

- `sid.freq_bt` -- Blackman-Tukey (fixed window, replaces `spa`)
- `sid.freq_btfdr` -- Blackman-Tukey with frequency-dependent resolution (replaces `spafdr`)
- `sid.freq_etfe` -- Empirical Transfer Function Estimate (replaces `etfe`)

Translated from `exampleMethodComparison.m`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter
import sid

%matplotlib inline

## Generate test data

First-order system with moderate noise.

In [ ]:
rng = np.random.default_rng(4)

N = 2000
u = rng.standard_normal(N)
y_clean = lfilter([1], [1, -0.85], u)
y = y_clean + 0.3 * rng.standard_normal(N)

## Estimate with all three methods

In [ ]:
r_bt     = sid.freq_bt(y, u, window_size=30)
r_etfe   = sid.freq_etfe(y, u)
r_etfe_s = sid.freq_etfe(y, u, smoothing=15)
r_fdr    = sid.freq_btfdr(y, u, resolution=0.3)

## Compare Bode magnitude plots

In [ ]:
w = r_bt.frequency
G_true = 1.0 / (1.0 - 0.85 * np.exp(-1j * w))

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(w, 20 * np.log10(np.abs(r_etfe.response)),
            color='0.8', label='ETFE (raw)')
ax.semilogx(w, 20 * np.log10(np.abs(r_etfe_s.response)),
            'g', label='ETFE (S=15)')
ax.semilogx(w, 20 * np.log10(np.abs(r_bt.response)),
            'b', label='BT (M=30)')
ax.semilogx(w, 20 * np.log10(np.abs(r_fdr.response)),
            'r', label='BTFDR (R=0.3)')
ax.semilogx(w, 20 * np.log10(np.abs(G_true)),
            'k--', lw=1.5, label='True')
ax.set_xlabel('Frequency (rad/sample)')
ax.set_ylabel('Magnitude (dB)')
ax.set_title('Method Comparison: Bode Magnitude')
ax.legend(loc='lower left')
ax.grid(True)
plt.tight_layout()
plt.show()

## Compare noise spectra

BT and BTFDR compute the noise spectrum from covariance estimates.
ETFE computes it from residuals.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(w, 10 * np.log10(np.abs(r_bt.noise_spectrum)),
            'b', label='BT')
ax.semilogx(w, 10 * np.log10(np.abs(r_etfe_s.noise_spectrum)),
            'g', label='ETFE (S=15)')
ax.semilogx(w, 10 * np.log10(np.abs(r_fdr.noise_spectrum)),
            'r', label='BTFDR')
ax.set_xlabel('Frequency (rad/sample)')
ax.set_ylabel('Noise Spectrum (dB)')
ax.set_title('Noise Spectrum Comparison')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## Custom logarithmic frequency grid

A log-spaced grid provides better low-frequency coverage.

In [ ]:
w_log = np.logspace(np.log10(0.05), np.log10(np.pi), 200)

r_bt_log   = sid.freq_bt(y, u, window_size=30, frequencies=w_log)
r_etfe_log = sid.freq_etfe(y, u, smoothing=15, frequencies=w_log)
r_fdr_log  = sid.freq_btfdr(y, u, resolution=0.3, frequencies=w_log)

G_true_log = 1.0 / (1.0 - 0.85 * np.exp(-1j * w_log))

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(w_log, 20 * np.log10(np.abs(r_bt_log.response)),
            'b', label='BT')
ax.semilogx(w_log, 20 * np.log10(np.abs(r_etfe_log.response)),
            'g', label='ETFE')
ax.semilogx(w_log, 20 * np.log10(np.abs(r_fdr_log.response)),
            'r', label='BTFDR')
ax.semilogx(w_log, 20 * np.log10(np.abs(G_true_log)),
            'k--', lw=1.5, label='True')
ax.set_xlabel('Frequency (rad/sample)')
ax.set_ylabel('Magnitude (dB)')
ax.set_title('Log Frequency Grid (200 Points)')
ax.legend(loc='lower left')
ax.grid(True)
plt.tight_layout()
plt.show()

## Time-series comparison: periodogram vs smoothed spectrum

With no input (`u=None`), all methods estimate the output power spectrum.

In [ ]:
rng5 = np.random.default_rng(5)

y_ts = lfilter([1], [1, -0.8], rng5.standard_normal(1000))

r_ts_bt   = sid.freq_bt(y_ts, None, window_size=30)
r_ts_etfe = sid.freq_etfe(y_ts, None)

w_ts = r_ts_bt.frequency
# True spectrum: |1/(1 - 0.8*e^{-jw})|^2
Phi_true = np.abs(1.0 / (1.0 - 0.8 * np.exp(-1j * w_ts)))**2

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(w_ts, 10 * np.log10(np.abs(r_ts_etfe.noise_spectrum)),
            color='0.7', label='ETFE (periodogram)')
ax.semilogx(w_ts, 10 * np.log10(np.abs(r_ts_bt.noise_spectrum)),
            'b', label='BT (M=30)')
ax.semilogx(w_ts, 10 * np.log10(Phi_true),
            'k--', lw=1.5, label='True')
ax.set_xlabel('Frequency (rad/sample)')
ax.set_ylabel('Power Spectrum (dB)')
ax.set_title('Time-Series: Periodogram vs Blackman-Tukey')
ax.legend(loc='lower left')
ax.grid(True)
plt.tight_layout()
plt.show()

## Model output comparison using `sid.compare`

Compare how well each method predicts the measured output.

In [ ]:
comp_bt   = sid.compare(r_bt,   y, u)
comp_fdr  = sid.compare(r_fdr,  y, u)
comp_etfe = sid.compare(r_etfe, y, u)

print('--- Prediction Fit (NRMSE %) ---')
print(f'  freq_bt:    {comp_bt.fit[0]:.1f}%')
print(f'  freq_btfdr: {comp_fdr.fit[0]:.1f}%')
print(f'  freq_etfe:  {comp_etfe.fit[0]:.1f}%')

## Summary of method trade-offs

| Method | Window Size | Uncertainty | Coherence |
|:-------|:------------|:------------|:----------|
| `freq_bt` | Fixed M | Yes | Yes |
| `freq_btfdr` | Per-freq $M_k$ | Yes | Yes |
| `freq_etfe` | N (full) | No | No |